# Badger_vision — Linux Training (Production-Ready)

**One notebook to train everything.** Point it at your dataset, pick a task, and go.

### Supported Dataset Formats

| Source | Layout | Task |
|--------|--------|------|
| Badger Factory – Object Detection | `yolo/images/{train,val}.7z` + `labels/{train,val}.7z` | Detection |
| Badger Factory – Keypoint Detection | Same YOLO layout | Keypoints |
| Badger Factory – Classification | `classifier/evolving_ds_*/train.7z` (class sub-folders) | Classification |
| COCO Export | `coco/{train,val}.7z` with `coco_instances.json` | Detection |
| Plain YOLO / COCO | Already extracted on disk | Any |

### Supported Archives
`.zip` · `.7z` · `.tar.gz` · `.tar.bz2` · `.tar.xz` · `.rar`

### Features
- Auto-setup: creates venv + installs everything if nothing is configured
- Always uses GPU (CUDA) — fails fast with a clear message if unavailable
- Progress bars (tqdm) for every batch
- Training snapshot every epoch: progress %, ETA, best loss, GPU memory
- Full checkpoint save/resume, EMA, AMP, early-stopping

---

## 1 — Environment Setup

This cell checks if a virtual environment is active and dependencies are
installed. If not, it creates a `.venv`, installs everything, and tells
you to restart the kernel.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Detect repo root
NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "pyproject.toml").exists() else NOTEBOOK_DIR

def in_venv():
    return sys.prefix != sys.base_prefix

if not in_venv():
    venv = REPO_ROOT / ".venv"
    if not venv.exists():
        print("Creating virtual environment ...")
        subprocess.check_call([sys.executable, "-m", "venv", str(venv)])
        pip = str(venv / "bin" / "pip")
        print("Installing Badger_vision + extras ...")
        subprocess.check_call([pip, "install", "--upgrade", "pip", "setuptools", "wheel"],
                              stdout=subprocess.DEVNULL)
        subprocess.check_call([pip, "install", "-e", str(REPO_ROOT)], stdout=subprocess.DEVNULL)
        subprocess.check_call([pip, "install", "tqdm", "py7zr", "rarfile", "jupyter", "ipykernel"],
                              stdout=subprocess.DEVNULL)
        # Register kernel
        python = str(venv / "bin" / "python")
        subprocess.check_call(
            [python, "-m", "ipykernel", "install", "--user",
             "--name", "badger_vision", "--display-name", "Badger_vision"],
            stdout=subprocess.DEVNULL,
        )
    print("\n" + "=" * 60)
    print("  Virtual environment ready at:", venv)
    print("  Switch the Jupyter kernel to 'Badger_vision' and re-run.")
    print("=" * 60)
else:
    # Ensure extras are installed
    for pkg in ["tqdm", "py7zr", "rarfile"]:
        try:
            __import__(pkg)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg],
                                  stdout=subprocess.DEVNULL)
    print("Environment OK:", sys.prefix)

## 2 — GPU Check

Training always runs on GPU. This cell verifies CUDA is available and
shows your GPU info.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No CUDA GPU detected!\n"
    "  - Check drivers:    nvidia-smi\n"
    "  - Check PyTorch:    python -c 'import torch; print(torch.version.cuda)'\n"
    "  - Install CUDA build: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121"
)

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"GPU  : {gpu_name}  ({gpu_mem:.1f} GB VRAM)")
print(f"CUDA : {torch.version.cuda}")
print(f"Torch: {torch.__version__}")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda:0")

## 3 — Configure Your Training

Settings are loaded from `train_config.yaml`. Edit that file to set your dataset path, task, model, epochs, etc.
You can also override values directly in the cell below.

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  Configuration — loaded from train_config.yaml               ║
# ╚═══════════════════════════════════════════════════════════════╝
# Edit train_config.yaml to change settings, or override below.

import yaml
from pathlib import Path

_cfg_path = Path("train_config.yaml")
if not _cfg_path.exists():
    _cfg_path = Path(__file__).resolve().parent / "train_config.yaml" if "__file__" in dir() else Path("train_config.yaml")

_tcfg = {}
if _cfg_path.exists():
    with open(_cfg_path) as _f:
        _tcfg = yaml.safe_load(_f) or {}
    print(f"Loaded config from {_cfg_path}")
else:
    print("train_config.yaml not found — using defaults")

DATASET_PATH = _tcfg.get("dataset_path", "/path/to/your/dataset")
TASK         = _tcfg.get("task", "detection")
EPOCHS       = _tcfg.get("epochs", 100)
BATCH_SIZE   = _tcfg.get("batch_size", 0)
IMG_SIZE     = _tcfg.get("img_size", 640)
LR           = _tcfg.get("lr", 0.01)
MODEL        = _tcfg.get("model", "resnext")
RESUME       = _tcfg.get("resume", None)

WORKSPACE    = "badger_vision_workspace"

## 4 — Extract & Prepare Dataset

This cell extracts archives (including nested per-split `.7z` files),
auto-detects the dataset format, and converts YOLO labels to COCO JSON
if needed.

In [ ]:
import shutil
from pathlib import Path

# Import helpers from the companion script
import importlib.util
spec = importlib.util.spec_from_file_location("linux_train", str(REPO_ROOT / "notebooks" / "linux_train.py"))
lt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lt)

workspace = Path(WORKSPACE).resolve()
workspace.mkdir(parents=True, exist_ok=True)

dataset_path = Path(DATASET_PATH).resolve()
assert dataset_path.exists(), f"Dataset not found: {dataset_path}"

# Extract top-level archive if needed
if dataset_path.is_file() and lt.is_archive(dataset_path):
    extract_dest = workspace / "dataset"
    if extract_dest.exists():
        shutil.rmtree(extract_dest)
    dataset_root = lt.extract_archive(dataset_path, extract_dest)
else:
    dataset_root = dataset_path

# Navigate to correct sub-folder for the task
dataset_root = lt.resolve_dataset_root(dataset_root, TASK)
print(f"Dataset root: {dataset_root}")

# Detect format & prepare
fmt = lt.detect_format(dataset_root)
print(f"Detected format: {fmt}")

if fmt == "badger_yolo":
    classes_txt = dataset_root / "classes.txt"
    data_info = lt.prepare_badger_yolo(dataset_root, workspace,
                                        classes_txt if classes_txt.exists() else None)
elif fmt == "badger_classifier":
    data_info = lt.prepare_badger_classifier(dataset_root, workspace)
elif fmt == "coco_archive":
    data_info = lt.prepare_coco_archive(dataset_root, workspace)
elif fmt == "yolo_flat":
    data_info = lt.prepare_yolo_flat(dataset_root, workspace)
elif fmt == "coco_flat":
    data_info = lt.prepare_coco_flat(dataset_root, workspace)
else:
    raise RuntimeError(
        f"Unknown dataset format in {dataset_root}\n"
        "Expected: Badger Factory YOLO, Classifier, COCO archive, or plain YOLO/COCO"
    )

num_classes = lt.detect_num_classes(data_info["train_ann_file"])
num_train   = lt.count_images(data_info["train_ann_file"])
print(f"Classes: {num_classes}  |  Training images: {num_train}")
if data_info.get("val_img_dir"):
    num_val = lt.count_images(data_info["val_ann_file"])
    print(f"Validation images: {num_val}")

## 5 — Auto Batch Size & Configs

In [ ]:
import torch

batch_size = BATCH_SIZE
if batch_size <= 0:
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    batch_size = lt.auto_batch_size(gpu_mem, IMG_SIZE)
    print(f"Auto batch size: {batch_size}  ({gpu_mem:.1f} GB VRAM)")

model_cfg, data_cfg = lt.write_configs(
    workspace, data_info, num_classes,
    epochs=EPOCHS, batch_size=batch_size,
    img_size=IMG_SIZE, lr=LR,
    model_type=MODEL,
)
print(f"Model config : {model_cfg}")
print(f"Data config  : {data_cfg}")

## 6 — Train!

Full training with:
- tqdm progress bar per batch
- Training snapshot every epoch (progress %, ETA, best loss, GPU mem)
- Checkpoints saved to `runs/train_<timestamp>/`
- Early stopping when loss plateaus

In [ ]:
lt.run_training(
    model_cfg_path=model_cfg,
    data_cfg_path=data_cfg,
    data_info=data_info,
    device=DEVICE,
    epochs=EPOCHS,
    batch_size=batch_size,
    resume=RESUME,
)

## 7 — Results

Load the best checkpoint and view final metrics.

In [ ]:
import glob
import torch

# Find the most recent run
run_dirs = sorted(glob.glob("runs/train_*"))
if run_dirs:
    latest = run_dirs[-1]
    best_ckpt = f"{latest}/checkpoint_best.pt"
    last_ckpt = f"{latest}/checkpoint_last.pt"

    ckpt_path = best_ckpt if Path(best_ckpt).exists() else last_ckpt
    if Path(ckpt_path).exists():
        ckpt = torch.load(ckpt_path, map_location="cpu")
        print(f"Checkpoint : {ckpt_path}")
        print(f"Epoch      : {ckpt.get('epoch', '?')}")
        print(f"Best Loss  : {ckpt.get('best_loss', '?'):.4f}")
    else:
        print("No checkpoint found.")
else:
    print("No training runs found.")